In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn as sk
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

pd.set_option('display.width', 3000)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)

# Dummy dataset loading
ml_dataset = pd.read_csv('pp.csv')  # Replace with your actual dataset path

print('Base data has %i rows and %i columns' % (ml_dataset.shape[0], ml_dataset.shape[1]))

ml_dataset = ml_dataset[['Calcium', 'Anion gap', 'SEX', 'Carbon dioxide', 'Hematocrit', 'Glucose', 'DISCH_DISP', 'Chloride', 'Lymphocytes', 'Phosphate', 'ASA_RATING', 'Hemoglobin', 'pH', 'Magnesium', 'WEIGHT', 'Creatinine', 'Neutrophils', 'HEIGHT', 'LOS', 'Monocytes', 'dx_name', 'Sodium', 'Urea nitrogen', 'Platelets', 'Potassium']]

def coerce_to_unicode(x):
    if sys.version_info < (3, 0):
        if isinstance(x, str):
            return unicode(x, 'utf-8')
        else:
            return unicode(x)
    else:
        return str(x)

categorical_features = ['SEX', 'DISCH_DISP', 'ASA_RATING', 'HEIGHT', 'dx_name']
numerical_features = ['Calcium', 'Anion gap', 'Carbon dioxide', 'Hematocrit', 'Glucose', 'Chloride', 'Lymphocytes', 'Phosphate', 'Hemoglobin', 'pH', 'Magnesium', 'WEIGHT', 'Creatinine', 'Neutrophils', 'Monocytes', 'Sodium', 'Urea nitrogen', 'Platelets', 'Potassium']

# Convert categorical features to unicode
for feature in categorical_features:
    ml_dataset[feature] = ml_dataset[feature].apply(coerce_to_unicode)

# Convert numerical features to double
for feature in numerical_features:
    ml_dataset[feature] = ml_dataset[feature].astype('double').round(2)

ml_dataset['__target__'] = ml_dataset['LOS']
del ml_dataset['LOS']

# Remove rows for which the target is unknown
ml_dataset = ml_dataset[~ml_dataset['__target__'].isnull()]

# Split into train and test datasets
train, test = train_test_split(ml_dataset, test_size=0.25, random_state=42)
print('Train data has %i rows and %i columns' % (train.shape[0], train.shape[1]))
print('Test data has %i rows and %i columns' % (test.shape[0], test.shape[1]))

# Impute missing values with mean for numerical features
for feature in numerical_features:
    train[feature] = train[feature].fillna(train[feature].mean()).round(2)
    test[feature] = test[feature].fillna(test[feature].mean()).round(2)
    print(f'Imputed missing values in feature {feature} with mean')

# Dummy encoding categorical features
LIMIT_DUMMIES = 100
categorical_to_dummy_encode = ['SEX', 'DISCH_DISP', 'ASA_RATING', 'HEIGHT', 'dx_name']

def select_dummy_values(train, features):
    dummy_values = {}
    for feature in categorical_to_dummy_encode:
        values = [
            value
            for (value, _) in Counter(train[feature]).most_common(LIMIT_DUMMIES)
        ]
        dummy_values[feature] = values
    return dummy_values

DUMMY_VALUES = select_dummy_values(train, categorical_to_dummy_encode)

def dummy_encode_dataframe(df):
    for (feature, dummy_values) in DUMMY_VALUES.items():
        for dummy_value in dummy_values:
            dummy_name = f'{feature}_value_{coerce_to_unicode(dummy_value)}'
            df[dummy_name] = (df[feature] == dummy_value).astype(float)
        del df[feature]
        print(f'Dummy-encoded feature {feature}')

dummy_encode_dataframe(train)
dummy_encode_dataframe(test)

# Rescale numerical features
rescale_features = {'Calcium': 'AVGSTD', 'Anion gap': 'AVGSTD', 'Carbon dioxide': 'AVGSTD', 'Hematocrit': 'AVGSTD', 'Glucose': 'AVGSTD', 'Chloride': 'AVGSTD', 'Lymphocytes': 'AVGSTD', 'Phosphate': 'AVGSTD', 'Hemoglobin': 'AVGSTD', 'pH': 'AVGSTD', 'Magnesium': 'AVGSTD', 'WEIGHT': 'AVGSTD', 'Creatinine': 'AVGSTD', 'Neutrophils': 'AVGSTD', 'Monocytes': 'AVGSTD', 'Sodium': 'AVGSTD', 'Urea nitrogen': 'AVGSTD', 'Platelets': 'AVGSTD', 'Potassium': 'AVGSTD'}

for (feature_name, rescale_method) in rescale_features.items():
    if rescale_method == 'MINMAX':
        _min = train[feature_name].min()
        _max = train[feature_name].max()
        scale = _max - _min
        shift = _min
    else:
        shift = train[feature_name].mean()
        scale = train[feature_name].std()
    if scale == 0.:
        del train[feature_name]
        del test[feature_name]
        print(f'Feature {feature_name} was dropped because it has no variance')
    else:
        train[feature_name] = ((train[feature_name] - shift) / scale).astype(np.float64).round(2)
        test[feature_name] = ((test[feature_name] - shift) / scale).astype(np.float64).round(2)
        print(f'Rescaled {feature_name}')

# Prepare training and testing sets
X_train = train.drop('__target__', axis=1)
X_test = test.drop('__target__', axis=1)
y_train = np.array(train['__target__'])
y_test = np.array(test['__target__'])



Base data has 320631 rows and 27 columns
Train data has 240473 rows and 25 columns
Test data has 80158 rows and 25 columns
Imputed missing values in feature Calcium with mean
Imputed missing values in feature Anion gap with mean
Imputed missing values in feature Carbon dioxide with mean
Imputed missing values in feature Hematocrit with mean
Imputed missing values in feature Glucose with mean
Imputed missing values in feature Chloride with mean
Imputed missing values in feature Lymphocytes with mean
Imputed missing values in feature Phosphate with mean
Imputed missing values in feature Hemoglobin with mean
Imputed missing values in feature pH with mean
Imputed missing values in feature Magnesium with mean
Imputed missing values in feature WEIGHT with mean
Imputed missing values in feature Creatinine with mean
Imputed missing values in feature Neutrophils with mean
Imputed missing values in feature Monocytes with mean
Imputed missing values in feature Sodium with mean
Imputed missing val

C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

Dummy-encoded feature HEIGHT


C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

Dummy-encoded feature dx_name
Dummy-encoded feature SEX
Dummy-encoded feature DISCH_DISP
Dummy-encoded feature ASA_RATING


C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

Dummy-encoded feature HEIGHT


C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[dummy_name] = (df[feature] == dummy_value).astype(float)
C:\Users\almas\AppData\Local\Temp\ipykernel_7820\3298103444.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

Dummy-encoded feature dx_name
Rescaled Calcium
Rescaled Anion gap
Rescaled Carbon dioxide
Rescaled Hematocrit
Rescaled Glucose
Rescaled Chloride
Rescaled Lymphocytes
Rescaled Phosphate
Rescaled Hemoglobin
Rescaled pH
Rescaled Magnesium
Rescaled WEIGHT
Rescaled Creatinine
Rescaled Neutrophils
Rescaled Monocytes
Rescaled Sodium
Rescaled Urea nitrogen
Rescaled Platelets
Rescaled Potassium


In [1]:
import joblib

# Load the model
model = joblib.load('day.pkl')


In [14]:
import numpy as np

# Replace these with your actual values
input_data = {
    'DISCH_DISP': 'Skilled Nursing Facility',  # This is the actual target, remove it before prediction
    'HEIGHT': "5' 0",
    'WEIGHT': 2299.84,  # In ounces
    'SEX': 'Female',
    'ASA_RATING': 'Severe Systemic Disease',
    'dx_name': 'Acute abdominal pain',
    'Anion gap': 8.166666667,
    'Calcium': 8.1125,
    'Carbon dioxide': 21.04166667,
    'Chloride': 110.4583333,
    'Creatinine': 1.256,
    'Glucose': 416791.375,
    'Hematocrit': 29.51904762,
    'Hemoglobin': 9.685714286,
    'Leukocytes^^corrected for nucleated erythrocytes': 15.93333333,
    'Lymphocytes': 2.569230769,
    'Magnesium': 6.93,
    'Monocytes': 0.938461538,
    'Neutrophils': 11.26153846,
    'Phosphate': 3.627777778,
    'Platelets': 287.9047619,
    'Potassium': 3.779166667,
    'Sodium': 139.6666667,
    'Urea nitrogen': 21.66666667,
    'pH': 7
}


# Convert the input data into a format suitable for the model (DataFrame or NumPy array)
input_values = np.array([list(input_data.values())]).reshape(1, -1)


In [15]:
# Make the prediction
predicted_los = model.predict(input_values)

# Display the result
print(f"Predicted Length of Stay (LOS): {predicted_los[0]}")


C:\Users\almas\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


ValueError: could not convert string to float: 'Skilled Nursing Facility'